# LLM Karşılaştırması — 100 Soru / 3 Qwen Modeli / Claude Hakem

**Sıra:** (1) Excel yükle → (2) 9B → (3) 27B → (4) 122B → (5) Claude ile puanla → (6) Excel indir. Her bölüm ayrı hücredir.

**Önemli:** 100 soru ve referans cevapları **taslaktır**; teslim etmeden kaynaklarını doğrula. Puanlar Claude’un rubrik puanlarıdır, doğruluk olasılığı değildir. **Tümünü Çalıştır** yerine hücreleri sırayla çalıştır. API anahtarını `OPENROUTER_API_KEY` adlı Colab Secrets alanından okur.


In [ ]:
!pip -q install openai pandas openpyxl

In [ ]:
from openai import OpenAI
from google.colab import userdata, files
import pandas as pd
import json

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))
print("API istemcisi hazır.")

# Hazır Excel'i bilgisayarından bir kez yükle
uploaded = files.upload()
dosya = next(name for name in uploaded if name.endswith(".xlsx"))
tablo = pd.read_excel(dosya, sheet_name="Yanitlar").fillna("")
puanlar = pd.read_excel(dosya, sheet_name="Degerlendirme").fillna("")
sampling = pd.DataFrame(columns=["ID", "Model", "Temperature", "Top-p", "Top-k", "Cevap", "Durum", "Not"])
def kaydet():
    with pd.ExcelWriter("LLM_Sade_Sonuclar.xlsx") as writer:
        tablo.to_excel(writer, sheet_name="Yanitlar", index=False)
        puanlar.to_excel(writer, sheet_name="Degerlendirme", index=False)
        sampling.to_excel(writer, sheet_name="Sampling", index=False)

print("Yüklenen soru:", len(tablo))

API istemcisi hazır.


Saving LLM_Sade_100_Soru_Tablo.xlsx to LLM_Sade_100_Soru_Tablo (1).xlsx
Yüklenen soru: 100


## Ortak kısa fonksiyon
Hocanın örneğindeki gibi `client.chat.completions.create` kullanıyoruz. Önceki `EMPTY` sorununa karşı düşünme kapalı, cevap sınırı 512. Boş sonuç olursa yanlış saymak yerine boş bırakıyoruz.

In [ ]:
def sor(model, soru, temperature=0.7, top_p=0.9, top_k=None):
    ek = {"reasoning": {"enabled": False}}
    if top_k is not None:
        ek["top_k"] = top_k
        ek["provider"] = {"require_parameters": True}
    r = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system", "content":"Türkçe kısa ve net cevap ver. Emin değilsen bilmiyorum de."},
            {"role":"user", "content":soru}
        ],
        temperature=temperature, top_p=top_p, max_tokens=512,
        extra_body=ek
    )
    return (r.choices[0].message.content or "").strip()

def modeli_calistir(model, sutun):
    tamam = 0
    for i, satir in tablo.iterrows():
        if str(tablo.at[i, sutun]).strip():
            continue
        try:
            cevap = sor(model, satir["Soru"])
            if not cevap:
                print(satir["ID"], "boş cevap; daha sonra yeniden denenebilir")
                continue
            tablo.at[i, sutun] = cevap
            tamam += 1
        except Exception as e:
            print(satir["ID"], "hata:", str(e)[:120])
        if (i + 1) % 20 == 0:
            kaydet()
            print(sutun, i + 1, "/", len(tablo))
    kaydet()
    print(sutun, "yeni yanıt:", tamam)

## Model 1 — Qwen3.5-9B (küçük)
Aynı soruları 9B modele gönderir; cevapları Excel’de **Qwen 9B** sütununa yerleştirir.

In [ ]:
modeli_calistir("qwen/qwen3.5-9b", "Qwen 9B")

Qwen 9B 20 / 100
Qwen 9B 40 / 100
Qwen 9B 60 / 100
Qwen 9B 80 / 100
Qwen 9B 100 / 100
Qwen 9B yeni yanıt: 100


## Model 2 — Qwen3.5-27B (orta)
**Qwen 27B** sütununu doldurur.

In [ ]:
modeli_calistir("qwen/qwen3.5-27b", "Qwen 27B")

Qwen 27B 20 / 100
Qwen 27B 40 / 100
Qwen 27B 60 / 100
Qwen 27B 80 / 100
Qwen 27B 100 / 100
Qwen 27B yeni yanıt: 100


## Model 3 — Qwen3.5-122B-A10B (büyük / MoE)
**Qwen 122B** sütununu doldurur. 122B toplam parametre; aynı anda tümü etkin değildir.

In [ ]:
modeli_calistir("qwen/qwen3.5-122b-a10b", "Qwen 122B")

Qwen 122B 20 / 100
Qwen 122B 40 / 100
Qwen 122B 60 / 100
Qwen 122B 80 / 100
Qwen 122B 100 / 100
Qwen 122B yeni yanıt: 100


## Sampling — hocanın temperature / top-p / top-k deneyi
İlk **3 soruda** üç farklı ayarı her modelle karşılaştırır (9 ek istek/model = 27). Ana 100 soruluk karşılaştırmayı değiştirmez. Burada sorun olursa yalnızca **Sampling** sayfasına hata kaydedilir.

In [ ]:

modeller = [
    "qwen/qwen3.5-9b",
    "qwen/qwen3.5-27b",
    "qwen/qwen3.5-122b-a10b"
]

ayarlar = [
    (0.0, 1.0, 1),
    (0.7, 0.9, 40),
    (1.25, 0.85, 50),
    (1.85, 1.0, 0)
]

sorular = tablo.head(4)[["ID", "Soru"]].values.tolist()
sorular.append([
    "SLOGAN",
    "7/24 acil tamirci uygulaması için 3 yaratıcı slogan yaz."
])

sampling = pd.DataFrame(columns=[
    "ID", "Soru", "Model", "Temperature",
    "Top-p", "Top-k", "Deneme",
    "Cevap", "Durum", "Not"
])

for soru_id, soru in sorular:
    for model in modeller:
        for temp, p, k in ayarlar:
            for deneme in range(1, 3):
                try:
                    cevap = sor(model, soru, temp, p, k)
                    durum = "OK" if cevap else "EMPTY"
                    hata = ""
                except Exception as e:
                    cevap = ""
                    durum = "ERROR"
                    hata = str(e)[:150]

                sampling.loc[len(sampling)] = [
                    soru_id, soru, model, temp, p, k,
                    deneme, cevap, durum, hata
                ]

                if len(sampling) % 10 == 0:
                    kaydet()
                    print(len(sampling), "/ 120 tamamlandı")

kaydet()
print("Sampling sonuçları kaydedildi.")


10 / 120 tamamlandı
20 / 120 tamamlandı
30 / 120 tamamlandı
40 / 120 tamamlandı
50 / 120 tamamlandı
60 / 120 tamamlandı
70 / 120 tamamlandı
80 / 120 tamamlandı
90 / 120 tamamlandı
100 / 120 tamamlandı
110 / 120 tamamlandı
120 / 120 tamamlandı
Sampling sonuçları kaydedildi.


In [ ]:
for cevap, puan, _ in sutunlar:
    print(
        cevap,
        "| Yanıt:", (tablo[cevap].astype(str).str.strip() != "").sum(),
        "| Claude puanı:", (puanlar[puan].astype(str).str.strip() != "").sum()
    )

print("Sampling kaydı:", len(sampling))

Qwen 9B | Yanıt: 100 | Claude puanı: 54
Qwen 27B | Yanıt: 100 | Claude puanı: 61
Qwen 122B | Yanıt: 100 | Claude puanı: 55
Sampling kaydı: 120


## Claude hakem — ayrı Excel sayfasına puan yazma
Referans cevaplarla her modelin ürettiği yanıtı karşılaştırır. 0–100 puan ve kısa gerekçe ister. `EMPTY` kayıtlar puanlanmaz; **sıfır puan verilmez**. Hakemin kararlarını teslimden önce örneklem üzerinde elle kontrol et.

In [ ]:
hakem = "anthropic/claude-haiku-4.5"
sutunlar = [("Qwen 9B", "9B Puan", "9B Gerekçe"),
            ("Qwen 27B", "27B Puan", "27B Gerekçe"),
            ("Qwen 122B", "122B Puan", "122B Gerekçe")]

for i, q in tablo.iterrows():
    for cevap_sutunu, puan_sutunu, gerekce_sutunu in sutunlar:
        cevap = str(q[cevap_sutunu]).strip()
        if not cevap or str(puanlar.at[i, puan_sutunu]).strip():
            continue
        metin = f"Soru: {q['Soru']}\nReferans: {q['Referans Cevap']}\nModel cevabı: {cevap}"
        try:
            r = client.chat.completions.create(
                model=hakem,
                messages=[
                    {"role":"system", "content": "Cevabı referansla karşılaştır. Eşdeğer ifadeleri doğru say. 0-100 puan ver. Yalnızca şu JSON'u yaz: {\"puan\": 0, \"gerekce\": \"kısa açıklama\"}"},
                    {"role":"user", "content":metin}
                ],
                temperature=0, max_tokens=160,
                extra_body={"reasoning": {"enabled": False}}
            )
            content = r.choices[0].message.content or ""
            content = content.replace("```json", "").replace("```", "").strip()
            deger = json.loads(content)
            puanlar.at[i, puan_sutunu] = max(0, min(100, float(deger["puan"])))
            puanlar.at[i, gerekce_sutunu] = deger["gerekce"]
        except Exception as e:
            print(q["ID"], cevap_sutunu, "hakem hatası:", str(e)[:100])
    if (i + 1) % 20 == 0:
        kaydet()
        print("Hakem:", i + 1, "/", len(tablo))
kaydet()
print("Hakem değerlendirmeleri tamamlandı / eksikler yeniden çalıştırılabilir.")

Hakem: 20 / 100
Q022 Qwen 122B hakem hatası: Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/anthropic/claude-4.5-h
Q023 Qwen 27B hakem hatası: Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/anthropic/claude-4.5-h
Q025 Qwen 122B hakem hatası: Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/anthropic/claude-4.5-h
Q028 Qwen 9B hakem hatası: Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/anthropic/claude-4.5-h
Q029 Qwen 27B hakem hatası: Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/anthropic/claude-4.5-h
Q029 Qwen 122B hakem hatası: Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/anthropic/claude-4.5-h
Q030 Qwen 9B hakem hatası: Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/anthropic/claude-4.5-h
Q031 Qwen 9B hakem hatası: Error code: 429 - {'error': {'message': 'Rate limit e

## Tek Excel çıktısı
Aynı dosyada **Yanitlar**, **Degerlendirme** ve **Sampling** sayfaları yer alır. Değerlendirme ve yanıtları kaydeder; dosyayı Colab'dan indir. Eksik puan veya yanıt varsa ilgili hücreyi tekrar çalıştırabilirsin.

In [ ]:
cikti = "LLM_Sade_Sonuclar.xlsx"
kaydet()

for cevap_col, puan_col, _ in sutunlar:
    print(cevap_col, "cevap:", (tablo[cevap_col].astype(str).str.strip() != "").sum(),
          "| puan:", (puanlar[puan_col].astype(str).str.strip() != "").sum())
files.download(cikti)

Qwen 9B cevap: 100 | puan: 77
Qwen 27B cevap: 100 | puan: 83
Qwen 122B cevap: 100 | puan: 83


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>